# 03 — Gold Layer
Read Silver Parquet. Build all aggregation tables that feed the dashboard.
Save each as a Gold Parquet. No raw data access, no cleaning here.

## Load Silver Parquet

In [16]:
import pandas as pd
import numpy as np

df       = pd.read_parquet("silver.parquet")
resolved = df.dropna(subset=["resolution_time_hours"])
csat_df  = df.dropna(subset=["csat_score"])

print("Total tickets:   ", len(df))
print("Resolved tickets:", len(resolved))
print("CSAT responses:  ", len(csat_df))

Total tickets:    82706
Resolved tickets: 49723
CSAT responses:   57949


## Resolution time by priority

In [17]:
priority_resolution = (
    resolved
    .groupby("priority")["resolution_time_hours"]
    .median()
    .reindex(["urgent", "high", "medium", "low"])
    .reset_index()
)
priority_resolution.columns = ["priority", "median_hours"]
print(priority_resolution)
priority_resolution.to_parquet("gold_priority_resolution.parquet", index=False)
print("Saved: gold_priority_resolution.parquet")

  priority  median_hours
0   urgent         4.875
1     high        14.730
2   medium        29.820
3      low        45.540
Saved: gold_priority_resolution.parquet


## Resolution time by SLA plan

In [18]:
sla_resolution = (
    resolved
    .groupby("sla_plan")["resolution_time_hours"]
    .median()
    .sort_values()
    .reset_index()
)
sla_resolution.columns = ["sla_plan", "median_hours"]
print(sla_resolution)
sla_resolution.to_parquet("gold_sla_resolution.parquet", index=False)
print("Saved: gold_sla_resolution.parquet")

   sla_plan  median_hours
0  platinum         29.15
1      gold         29.73
2  standard         30.01
Saved: gold_sla_resolution.parquet


## Monthly ticket volume

In [19]:
monthly = (
    df.groupby("year_month")
    .agg(n_tickets=("ticket_id", "count"), avg_csat=("csat_score", "mean"))
    .reset_index()
    .sort_values("year_month")
)
print(monthly.head())
monthly.to_parquet("gold_monthly.parquet", index=False)
print("Saved: gold_monthly.parquet")

  year_month  n_tickets  avg_csat
0    2022-01       1738  3.180500
1    2022-02       1536  3.122355
2    2022-03       1747  3.183740
3    2022-04       1723  3.224700
4    2022-05       1663  3.279931
Saved: gold_monthly.parquet


## Status breakdown

In [20]:
status_breakdown = df["status"].value_counts().reset_index()
status_breakdown.columns = ["status", "count"]
print(status_breakdown)
status_breakdown.to_parquet("gold_status.parquet", index=False)
print("Saved: gold_status.parquet")

             status  count
0          resolved  41442
1       in_progress  16315
2           on_hold   8413
3  closed_no_action   8281
4              open   8255
Saved: gold_status.parquet


## CSAT by SLA plan

In [21]:
csat_by_sla = (
    csat_df.groupby("sla_plan")["csat_score"]
    .mean()
    .sort_values()
    .reset_index()
)
csat_by_sla.columns = ["sla_plan", "csat_score"]
print(csat_by_sla)
csat_by_sla.to_parquet("gold_csat_sla.parquet", index=False)
print("Saved: gold_csat_sla.parquet")

   sla_plan  csat_score
0      gold    3.190357
1  standard    3.202053
2  platinum    3.209040
Saved: gold_csat_sla.parquet


## Resolution time by region and SLA

In [22]:
region_sla = (
    resolved
    .groupby(["region", "sla_plan"])["resolution_time_hours"]
    .median()
    .unstack()
    .reset_index()
)
print(region_sla)
region_sla.to_parquet("gold_region_sla.parquet", index=False)
print("Saved: gold_region_sla.parquet")

sla_plan region    gold  platinum  standard
0          APAC  30.650    30.690    30.245
1            EU  28.600    28.120    29.880
2         LATAM  29.710    27.535    29.620
3           MEA  30.385    31.095    30.525
4            NA  29.470    28.740    27.190
Saved: gold_region_sla.parquet


## Ticket volume by channel and issue type

In [23]:
channel_counts = df["channel"].value_counts().reset_index()
channel_counts.columns = ["channel", "count"]

issue_counts = df["issue_type"].value_counts().reset_index()
issue_counts.columns = ["issue_type", "count"]

print(channel_counts)
print(issue_counts)
channel_counts.to_parquet("gold_channel.parquet", index=False)
issue_counts.to_parquet("gold_issue_type.parquet", index=False)
print("Saved: gold_channel.parquet, gold_issue_type.parquet")

            channel  count
0             email  16638
1  phone_transcript  16635
2            in_app  16548
3          web_form  16524
4              chat  16361
         issue_type  count
0            how_to  10513
1    account_access  10411
2       performance  10405
3             other  10312
4   billing_problem  10305
5  security_concern  10303
6   feature_request  10302
7               bug  10155
Saved: gold_channel.parquet, gold_issue_type.parquet


## Ticket volume by product area

In [24]:
product_counts = df["product_area"].value_counts().reset_index()
product_counts.columns = ["product_area", "count"]
print(product_counts)
product_counts.to_parquet("gold_product_area.parquet", index=False)
print("Saved: gold_product_area.parquet")

          product_area  count
0      api_integration  11996
1           mobile_app  11921
2           login_auth  11886
3              billing  11770
4  analytics_dashboard  11750
5          data_export  11721
6        notifications  11662
Saved: gold_product_area.parquet


## CSAT heatmap (segment × region)

In [25]:
csat_heatmap = (
    csat_df
    .groupby(["customer_segment", "region"])["csat_score"]
    .mean()
    .unstack()
    .reset_index()
)
print(csat_heatmap)
csat_heatmap.to_parquet("gold_csat_heatmap.parquet", index=False)
print("Saved: gold_csat_heatmap.parquet")

region customer_segment      APAC        EU     LATAM       MEA        NA
0             education  3.206514  3.200967  3.260406  3.183397  3.330275
1            enterprise  3.198705  3.201054  3.168036  3.232105  3.310000
2            individual  3.194081  3.229145  3.225299  3.212153  3.241758
3            non_profit  3.188507  3.174124  3.204513  3.168999  3.168317
4        small_business  3.182732  3.191762  3.186119  3.167244  3.295455
Saved: gold_csat_heatmap.parquet


## Issue type stats — CSAT, reopen rate, resolution time

In [26]:
issue_stats = (
    df.groupby("issue_type")
    .agg(
        avg_csat        = ("csat_score",            "mean"),
        reopen_rate     = ("reopened",               "mean"),
        median_res_time = ("resolution_time_hours",  "median"),
    )
    .sort_values("avg_csat")
    .reset_index()
)
print(issue_stats)
issue_stats.to_parquet("gold_issue_stats.parquet", index=False)
print("Saved: gold_issue_stats.parquet")

         issue_type  avg_csat  reopen_rate  median_res_time
0  security_concern  2.799200     0.052509           29.810
1    account_access  2.810515     0.053213           29.505
2   billing_problem  2.821424     0.048229           30.370
3       performance  2.828194     0.053051           29.695
4               bug  3.420095     0.051896           29.180
5             other  3.438238     0.051978           29.360
6   feature_request  3.725764     0.050184           30.515
7            how_to  3.754712     0.046989           30.535
Saved: gold_issue_stats.parquet


## Sentiment distribution

In [27]:
sentiment_counts = (
    df["customer_sentiment"]
    .value_counts()
    .reindex(["very_negative", "negative", "neutral", "positive", "very_positive"])
    .reset_index()
)
sentiment_counts.columns = ["customer_sentiment", "count"]
print(sentiment_counts)
sentiment_counts.to_parquet("gold_sentiment.parquet", index=False)
print("Saved: gold_sentiment.parquet")

  customer_sentiment  count
0      very_negative  13613
1           negative  21490
2            neutral  26957
3           positive  13445
4      very_positive   7201
Saved: gold_sentiment.parquet


## CSAT score distribution (1–5 bar chart)

In [28]:
csat_dist = (
    csat_df["csat_score"]
    .value_counts()
    .sort_index()
    .reset_index()
)
csat_dist.columns = ["csat_score", "count"]
print(csat_dist)
csat_dist.to_parquet("gold_csat_dist.parquet", index=False)
print("Saved: gold_csat_dist.parquet")

   csat_score  count
0         1.0   4823
1         2.0  12322
2         3.0  16888
3         4.0  14315
4         5.0   9601
Saved: gold_csat_dist.parquet


## Headline metrics

In [29]:
metrics = {
    "total_tickets":          len(df),
    "resolution_rate_pct":    round(df["is_resolved"].mean() * 100, 1),
    "median_resolution_hours":round(resolved["resolution_time_hours"].median(), 1),
    "mean_resolution_hours":  round(resolved["resolution_time_hours"].mean(), 1),
    "reopen_rate_pct":        round(df["reopened"].mean() * 100, 1),
    "csat_response_rate_pct": round(len(csat_df) / len(df) * 100, 1),
    "avg_csat":               round(csat_df["csat_score"].mean(), 2),
    "unresolved_backlog":     int((~df["is_resolved"]).sum()),
}
for k, v in metrics.items():
    print(f"  {k}: {v}")

pd.DataFrame([metrics]).to_parquet("gold_metrics.parquet", index=False)
print("Saved: gold_metrics.parquet")

  total_tickets: 82706
  resolution_rate_pct: 60.1
  median_resolution_hours: 29.9
  mean_resolution_hours: 45.1
  reopen_rate_pct: 5.1
  csat_response_rate_pct: 70.1
  avg_csat: 3.2
  unresolved_backlog: 32983
Saved: gold_metrics.parquet


## Confirm all Gold files

In [30]:
import os
gold_files = sorted([f for f in os.listdir(".") if f.startswith("gold_")])
print(f"{len(gold_files)} Gold files saved:")
for f in gold_files:
    print(" ", f)

14 Gold files saved:
  gold_channel.parquet
  gold_csat_dist.parquet
  gold_csat_heatmap.parquet
  gold_csat_sla.parquet
  gold_issue_stats.parquet
  gold_issue_type.parquet
  gold_metrics.parquet
  gold_monthly.parquet
  gold_priority_resolution.parquet
  gold_product_area.parquet
  gold_region_sla.parquet
  gold_sentiment.parquet
  gold_sla_resolution.parquet
  gold_status.parquet
